# Ouroboros — generate the 1M dataset (Phase 1)

Builds pool -> composition -> shards for 1M training molecules (the 200k set is its first 200 shards) plus the shared 5k val / 10k test sets, then copies everything to Drive. CPU-only work: any runtime with many vCPUs works; expected ~35-60 min on 12 vCPUs, ~7 GB of shards.

All project code runs in subprocesses (`!python ...`) so the pinned numpy etc. take effect without restarting the kernel. If the repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repository.

In [ ]:
import time, os, subprocess, json
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true
!python --version && nproc && free -g | head -2

In [ ]:
# ---- configuration ----
REPO = 'yaniguan/ouroboros-ocsr'
BRANCH = 'claude/vigilant-johnson-j4882f'  # set to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/ouroboros'  # data, runs, results live here
REPO_DIR = '/content/ouroboros-ocsr'
LOCAL_DATA = '/content/data/full'  # configs expect shards at /content/data/full/shards

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for sub in ('data', 'runs', 'results', 'real'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log --oneline -1

In [ ]:
%%bash -s "$REPO_DIR"
set -e
cd "$1"
# py3nj (escnn dependency) builds from source and needs a Fortran compiler
which gfortran || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
pip install -q -r requirements-colab.txt
pip install -q --no-deps -e .
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
t = time.time()
!python scripts/build_dataset.py --out {LOCAL_DATA} --cache /content/data_cache --preset 1m --workers $(nproc)
GEN_MIN = (time.time() - t) / 60
print(f'generation wall time: {GEN_MIN:.1f} min')

In [ ]:
t = time.time()
!mkdir -p {DRIVE_ROOT}/data/full
!rsync -a --info=progress2 {LOCAL_DATA}/ {DRIVE_ROOT}/data/full/
print(f'copy to Drive: {(time.time() - t) / 60:.1f} min')

In [ ]:
# ---- report these numbers back ----
!du -sh {LOCAL_DATA}/shards && ls {LOCAL_DATA}/shards/train-*.tar | wc -l
!du -ch {LOCAL_DATA}/shards/train-000[01]*.tar | tail -1   # = 200k subset size
!cat {LOCAL_DATA}/pool.stats.json {LOCAL_DATA}/manifests/compose.json
!cat {LOCAL_DATA}/shards/*.render.jsonl
print(f'generation {GEN_MIN:.1f} min; total notebook {(time.time() - T0) / 60:.1f} min')